# **Setup**

In [1]:
import os, sys

# Repository information
REPO_NAME = "RecSys-Challenge-2025"
REPO_URL  = f"github.com/Lv1g1/{REPO_NAME}.git"

# Detect environment
IS_COLAB = 'content' in os.getcwd()
IS_KAGGLE = 'kaggle' in os.getcwd()
IS_LOCAL = not (IS_COLAB or IS_KAGGLE)

WORKING_DIR = os.getcwd()

if IS_COLAB:
    WORKING_DIR = "/content"

    # Mount Google Drive
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)

    # Get GitHub token via input
    def get_token():
        from getpass import getpass
        return getpass("GitHub Token: ")

elif IS_KAGGLE:
    WORKING_DIR = "/kaggle/working"

    # Get GitHub token from Kaggle secrets
    def get_token():
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("Token")

# If local environment assume inside the repo
LOCAL_REPO_PATH = "/home/luigi/RecSys" if IS_LOCAL else os.path.join(WORKING_DIR, REPO_NAME)

# Clone the repository if it doesn't exist
if not os.path.exists(LOCAL_REPO_PATH):
    os.chdir(WORKING_DIR)
    token = get_token()

    !git clone https://{token}@{REPO_URL}
else:
    print("Repo already exists — pulling latest changes")
    os.chdir(LOCAL_REPO_PATH)
    !git pull
    os.chdir(WORKING_DIR)

# Add to Python PATH
if LOCAL_REPO_PATH not in sys.path:
    sys.path.append(LOCAL_REPO_PATH)

Repo already exists — pulling latest changes
Already up to date.


In [2]:
if IS_COLAB or False:  # Set to True if you want to recompile Cython files
    os.chdir(LOCAL_REPO_PATH)
    !python run_compile_all_cython.py
    os.chdir(WORKING_DIR)

In [3]:
if IS_COLAB or IS_KAGGLE:
    !pip install optuna

import optuna

In [4]:
import importlib
import numpy as np

from Challenge import paths
importlib.reload(paths)

from Challenge.hyper_tuning import ModelOptimizer

Running on local — storage at: /home/luigi/RecSys
Running on local — storage at: /home/luigi/RecSys


# **Load data**

In [5]:
# Load datasets
folds = paths.load_cv_folds(k=5)

In [6]:
def evaluate_recommender(recommender, at, URM_validation):
    cumulative_recall = 0.0
    num_eval = 0
    
    for user_id in range(URM_validation.shape[0]):
        relevant_items = URM_validation.indices[URM_validation.indptr[user_id]:URM_validation.indptr[user_id+1]]
        
        if len(relevant_items)>0:
            num_eval+=1
            
            recommended_items = recommender.recommend(user_id, cutoff=at)
            
            is_relevant = np.isin(recommended_items, relevant_items, assume_unique=True)
            recall_score = np.sum(is_relevant, dtype=np.float32) / relevant_items.shape[0]

            cumulative_recall += recall_score

    return cumulative_recall / num_eval

# **Train a KNN with Cosine similarity**

## **Hyperparameter Tuning**

In [7]:
from Recommenders.KNN.ItemKNNCFRecommender import ItemKNNCFRecommender

optimizer = ModelOptimizer("ItemKNN_cosine")

STUDY_NAME = ItemKNNCFRecommender.RECOMMENDER_NAME + "_cosine_a"

In [8]:
def objective_function(optuna_trial: optuna.trial.Trial) -> float:
    params = {
        "similarity": "cosine",
        "topK": optuna_trial.suggest_int("topK", 10, 1500),
        "shrink": optuna_trial.suggest_int("shrink", 0, 2000),
        "normalize": optuna_trial.suggest_categorical("normalize", [True, False]),
        "feature_weighting": optuna_trial.suggest_categorical("feature_weighting", ["BM25", "TF-IDF", "none"])
    }
    
    validation_scores = []
    for URM_train, URM_validation in folds:        
        # Train the recommender
        recommender_instance = ItemKNNCFRecommender(URM_train)
        recommender_instance.fit(**params)
        
        # Evaluate
        score = evaluate_recommender(recommender_instance, at=20, URM_validation=URM_validation)
        validation_scores.append(score)
        
        # Show fold result
        print(f"  Fold {len(validation_scores)} - Score: {score}")

        # Report intermediate result to Optuna
        optuna_trial.report(np.mean(validation_scores), len(validation_scores))

        # Ask Optuna to prune if performance is poor
        if optuna_trial.should_prune():
            raise optuna.TrialPruned()
        
        # Log fold performance
        optimizer.log_fold_performance(len(validation_scores), score)

    return np.mean(validation_scores)

In [9]:
optuna_study = optimizer.create_and_optimize_study(
    study_name=STUDY_NAME,
    objective_function=objective_function,
    n_trials=40
)

[I 2025-11-09 22:45:09,385] Using an existing study with name 'ItemKNNCFRecommender_cosine_a' instead of creating a new one.


  0%|          | 0/40 [00:00<?, ?it/s]

Similarity column 6969 (100.0%), 4631.94 column/sec. Elapsed time 1.50 sec
  Fold 1 - Score: 0.2112681120634079
Baseline Scores at step 1 : [0.21271030604839325, 0.21028369665145874, 0.21013639867305756] 0.2112681120634079
Similarity column 6969 (100.0%), 4772.14 column/sec. Elapsed time 1.46 sec
  Fold 2 - Score: 0.21299855411052704
Baseline Scores at step 2 : [0.21351823210716248, 0.2109064757823944, 0.2107119858264923] 0.21213333308696747
Similarity column 6969 (100.0%), 4731.03 column/sec. Elapsed time 1.47 sec
  Fold 3 - Score: 0.2139546424150467
Baseline Scores at step 3 : [0.21435575187206268, 0.21133679151535034, 0.21143142879009247] 0.2127404361963272
Similarity column 6969 (100.0%), 4732.07 column/sec. Elapsed time 1.47 sec
  Fold 4 - Score: 0.21267756819725037
Baseline Scores at step 4 : [0.21430015563964844, 0.2113673985004425, 0.21139031648635864] 0.2127247154712677
Similarity column 6969 (100.0%), 4669.90 column/sec. Elapsed time 1.49 sec
  Fold 5 - Score: 0.2144188135862

KeyboardInterrupt: 

In [ ]:
optuna.visualization.plot_optimization_history(optuna_study)

In [ ]:
optuna.visualization.plot_param_importances(optuna_study)

In [ ]:
optuna.visualization.plot_parallel_coordinate(optuna_study)

In [ ]:
bp = optimizer.get_best_params()

def objective_function(optuna_trial: optuna.trial.Trial) -> float:
    params = {
        "similarity": "cosine",
        "topK": optuna_trial.suggest_int("topK", max(1, bp["topK"] - 100), bp["topK"] + 100),
        "shrink": optuna_trial.suggest_int("shrink", max(0, bp["shrink"] - 100), bp["shrink"] + 100),
        "normalize": bp["normalize"],
        "feature_weighting": bp["feature_weighting"]
    }
    
    validation_scores = []
    for URM_train, URM_validation in folds:        
        # Train the recommender
        recommender_instance = ItemKNNCFRecommender(URM_train)
        recommender_instance.fit(**params)
        
        # Evaluate
        score = evaluate_recommender(recommender_instance, at=20, URM_validation=URM_validation)
        validation_scores.append(score)
        
        # Show fold result
        print(f"  Fold {len(validation_scores)} - Score: {score}")

        # Report intermediate result to Optuna
        optuna_trial.report(np.mean(validation_scores), len(validation_scores))

        # Ask Optuna to prune if performance is poor
        if optuna_trial.should_prune():
            raise optuna.TrialPruned()
        
        # Log fold performance
        optimizer.log_fold_performance(len(validation_scores), score)

    return np.mean(validation_scores)

In [ ]:
optuna_study = optimizer.create_and_optimize_study(
    study_name=STUDY_NAME,
    objective_function=objective_function,
    n_trials=40
)

# **Best Params**

- ADD HERE